# PULSE — Source Substrate Audit

**Purpose**: Exhaustive pass over every sheet/column/row of the 3 xlsx files + qualitative pass on the 2 PDFs and 1 DOCX.

**Outputs**:
- A substrate dossier documenting every field, pattern, and anomaly
- Drives `prompts/heuristic_keywords.yaml` (keyword dictionary)
- Drives `agents/normalization/taxonomies.py` (canonical enums)
- Informs `prompts/uncertainty.yaml` initial parameters

This is the most important notebook. Run it before anything else.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import json

RAW_DATA = ROOT / 'raw_data'
MANIFEST = json.loads((RAW_DATA / 'manifest.json').read_text())

print('Source files:')
for name, sha in MANIFEST.items():
    path = RAW_DATA / name
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f'  {name}: {sha[:16]}... ({size_kb:.1f} KB)')

## 1. Fund_Rating_Guide.xlsx

In [ ]:
xl_rating = pd.ExcelFile(RAW_DATA / 'Fund_Rating_Guide.xlsx', engine='openpyxl')
print('Sheets:', xl_rating.sheet_names)

for sheet in xl_rating.sheet_names:
    df = xl_rating.parse(sheet, dtype=str)
    print(f'\n--- Sheet: {sheet} ({df.shape[0]} rows × {df.shape[1]} cols) ---')
    print('Columns:', list(df.columns))
    display(df.head(5))

## 2. MyAsiaVC LP Scoping.xlsx

In [ ]:
xl_scoping = pd.ExcelFile(RAW_DATA / 'MyAsiaVC LP Scoping.xlsx', engine='openpyxl')
print('Sheets:', xl_scoping.sheet_names)

for sheet in xl_scoping.sheet_names:
    df = xl_scoping.parse(sheet, dtype=str)
    print(f'\n--- Sheet: {sheet} ({df.shape[0]} rows × {df.shape[1]} cols) ---')
    print('Columns:', list(df.columns))
    display(df.head(10))
    
    # Value distributions for key columns
    for col in df.columns[:8]:
        if df[col].nunique() < 30:
            print(f'\n  {col} value counts:')
            print(df[col].value_counts().head(10))

## 3. MyAsiaVC_ICP_4.0_Prospect_List_External.xlsx

In [ ]:
xl_icp = pd.ExcelFile(RAW_DATA / 'MyAsiaVC_ICP_4.0_Prospect_List_External.xlsx', engine='openpyxl')
print('Sheets:', xl_icp.sheet_names)

for sheet in xl_icp.sheet_names:
    df = xl_icp.parse(sheet, dtype=str)
    print(f'\n--- Sheet: {sheet} ({df.shape[0]} rows × {df.shape[1]} cols) ---')
    print('Columns:', list(df.columns))
    display(df.head(10))

## 4. PDF Substrate — Fund_Pre Screening Briefing_Call_Prep.pdf

In [ ]:
import pdfplumber

pdf_path = RAW_DATA / 'Fund_Pre Screening Briefing_Call_Prep.pdf'
print(f'File: {pdf_path.name}')

with pdfplumber.open(pdf_path) as pdf:
    print(f'Pages: {len(pdf.pages)}')
    for i, page in enumerate(pdf.pages, 1):
        text = page.extract_text() or ''
        tables = page.extract_tables() or []
        print(f'\n--- Page {i} ({len(text)} chars, {len(tables)} tables) ---')
        if tables:
            for t in tables:
                print('  Table:', t[:3])
        if text.strip():
            print(text[:500])

## 5. PDF Substrate — LP Side Plan Draft 1.pdf

In [ ]:
pdf_path2 = RAW_DATA / 'LP Side Plan Draft 1.pdf'

with pdfplumber.open(pdf_path2) as pdf:
    print(f'Pages: {len(pdf.pages)}')
    for i, page in enumerate(pdf.pages, 1):
        text = page.extract_text() or ''
        print(f'\n--- Page {i} ({len(text)} chars) ---')
        if text.strip():
            print(text[:600])

## 6. DOCX Substrate — AI_Native_VC_Fund_Strategy.docx

In [ ]:
import docx

doc_path = RAW_DATA / 'AI_Native_VC_Fund_Strategy.docx'
doc = docx.Document(str(doc_path))

print(f'Paragraphs: {len(doc.paragraphs)}')
headings = [(p.style.name, p.text) for p in doc.paragraphs if p.style.name.lower().startswith('heading')]
print(f'\nHeadings ({len(headings)}):')
for style, text in headings:
    print(f'  [{style}] {text}')

print('\nAll paragraphs (first 50):')
for i, p in enumerate(doc.paragraphs[:50]):
    if p.text.strip():
        print(f'  {i}: [{p.style.name}] {p.text[:120]}')

## 7. Cross-file Column Inventory

Map all column names across the 3 xlsx files to understand overlap and normalization needs.

In [ ]:
all_columns = {}

for fname, xl in [
    ('Fund_Rating_Guide', xl_rating),
    ('LP_Scoping', xl_scoping),
    ('ICP_4.0', xl_icp),
]:
    for sheet in xl.sheet_names:
        df = xl.parse(sheet, dtype=str)
        for col in df.columns:
            col_norm = col.strip().lower()
            all_columns.setdefault(col_norm, []).append(f'{fname}:{sheet}')

print('Column name → files where it appears:')
for col, files in sorted(all_columns.items()):
    print(f'  "{col}" → {files}')

## 8. Entity Name Candidates

Extract candidate LP/fund names from each sheet — baseline for the entity resolver.

In [ ]:
name_fields = ['LP Name', 'Name', 'Investor Name', 'Allocator', 'Institution', 'Organization', 'Entity', 'Company', 'Fund Name', 'LP']

all_names = []
for fname, xl in [('LP_Scoping', xl_scoping), ('ICP_4.0', xl_icp), ('Rating', xl_rating)]:
    for sheet in xl.sheet_names:
        df = xl.parse(sheet, dtype=str)
        df.columns = [c.strip() for c in df.columns]
        for field in name_fields:
            if field in df.columns:
                names = df[field].dropna().unique()
                for n in names:
                    n = str(n).strip()
                    if n and n not in ('nan', 'None', ''):
                        all_names.append({'name': n, 'source': f'{fname}:{sheet}:{field}'})

print(f'Total entity name candidates: {len(all_names)}')
for item in all_names[:30]:
    print(f'  {item["name"]} [{item["source"]}]')

## 9. Substrate Dossier Summary

Document key findings for taxonomy and keyword dictionary configuration.

In [ ]:
dossier = {
    "xlsx_files": [
        "Fund_Rating_Guide.xlsx",
        "MyAsiaVC LP Scoping.xlsx",
        "MyAsiaVC_ICP_4.0_Prospect_List_External.xlsx",
    ],
    "prose_files": [
        "Fund_Pre Screening Briefing_Call_Prep.pdf",
        "LP Side Plan Draft 1.pdf",
        "AI_Native_VC_Fund_Strategy.docx",
    ],
    "total_entity_name_candidates": len(all_names),
    "unique_column_names_across_sheets": len(all_columns),
    "notes": [
        "Run this cell after reviewing outputs above to populate the dossier.",
        "Add findings about: allocator types observed, geography clusters, rejection language, committee patterns.",
        "These findings drive heuristic_keywords.yaml and taxonomies.py.",
    ]
}

import json
dossier_path = ROOT / 'docs' / 'substrate_dossier.json'
dossier_path.parent.mkdir(parents=True, exist_ok=True)
dossier_path.write_text(json.dumps(dossier, indent=2))
print(f'Substrate dossier written to {dossier_path}')
print(json.dumps(dossier, indent=2))